# 🔬 Notebook 3 — Stock Exchange: Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Then **select the `.venv` kernel** in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses only the Python standard library + `pydantic` — nothing else to install, no Docker required.


## What we'll build

Three short deep-dives, each as a **bad → best** pair of runnable cells:

1. **Matching engine** — from a naive list scan to a price-level book with FIFO queues.
2. **Determinism & replay** — how a WAL lets us rebuild state after a crash.
3. **Market-data fan-out** — why the matcher must **never** push to subscribers directly.

You can run each pair in isolation.

## 1. Matching engine — price-time priority

**Price-time priority**: at any given price level, the order that arrived **first** is matched first. It is the fairness rule that every major exchange uses.

### Bad version: scan a flat list

Works, but every incoming order does an O(n) scan over all resting orders. With 1 M resting orders this is unusable.

In [1]:
from dataclasses import dataclass, field
from itertools import count
import time, random

@dataclass
class Order:
    id: int
    side: str           # 'buy' | 'sell'
    price: int          # integer ticks — see Notebook 2
    qty: int
    ts: int = field(default_factory=count().__next__)   # arrival order

def match_bad(resting: list[Order], incoming: Order):
    trades = []
    # BAD: linear scan, plus we must re-sort every time to find the best price
    opposite = "sell" if incoming.side == "buy" else "buy"
    candidates = [o for o in resting if o.side == opposite]
    # price-time priority: best price first, oldest first
    candidates.sort(key=lambda o: (o.price if incoming.side == "buy" else -o.price, o.ts))
    for o in candidates:
        if incoming.qty == 0: break
        crosses = (incoming.side == "buy"  and o.price <= incoming.price) or \
                  (incoming.side == "sell" and o.price >= incoming.price)
        if not crosses: break
        fill = min(o.qty, incoming.qty)
        trades.append((incoming.id, o.id, o.price, fill))
        o.qty -= fill; incoming.qty -= fill
    resting[:] = [o for o in resting if o.qty > 0]
    if incoming.qty > 0:
        resting.append(incoming)
    return trades

# Micro-benchmark: 5,000 resting + 5,000 incoming
random.seed(0)
book = [Order(id=i, side="sell", price=100 + random.randint(0, 20), qty=10)
        for i in range(5_000)]
start = time.perf_counter()
for i in range(5_000):
    match_bad(book, Order(id=100_000+i, side="buy", price=110, qty=1))
print(f"BAD matcher: {time.perf_counter()-start:.3f}s for 10k events")

BAD matcher: 3.129s for 10k events


### Best version: per-price FIFO queues

Two dicts `bids[price]` and `asks[price]`, each mapping to a `deque` of orders in arrival order. The best price on each side is the dict's min/max key. Matching pops from the front of a deque — O(1) — and we only ever touch the *best* price level.

We also support **cancel** and **partial fills** correctly.

In [2]:
from collections import defaultdict, deque

class MatchingEngine:
    def __init__(self):
        self.bids: dict[int, deque[Order]] = defaultdict(deque)   # price → FIFO
        self.asks: dict[int, deque[Order]] = defaultdict(deque)
        self.index: dict[int, Order] = {}                         # order id → Order (for cancel)

    # ----- helpers ---------------------------------------------------
    def _best_ask(self): return min(self.asks) if self.asks else None
    def _best_bid(self): return max(self.bids) if self.bids else None

    def _rest(self, o):
        (self.bids if o.side == "buy" else self.asks)[o.price].append(o)
        self.index[o.id] = o

    # ----- public API ------------------------------------------------
    def submit(self, o: Order):
        trades = []
        if o.side == "buy":
            while o.qty and self.asks and (best := self._best_ask()) <= o.price:
                q = self.asks[best]
                top = q[0]
                fill = min(top.qty, o.qty)
                trades.append((o.id, top.id, best, fill))
                top.qty -= fill; o.qty -= fill
                if top.qty == 0:
                    q.popleft(); self.index.pop(top.id, None)
                    if not q: del self.asks[best]
        else:
            while o.qty and self.bids and (best := self._best_bid()) >= o.price:
                q = self.bids[best]
                top = q[0]
                fill = min(top.qty, o.qty)
                trades.append((top.id, o.id, best, fill))
                top.qty -= fill; o.qty -= fill
                if top.qty == 0:
                    q.popleft(); self.index.pop(top.id, None)
                    if not q: del self.bids[best]
        if o.qty:
            self._rest(o)
        return trades

    def cancel(self, order_id: int) -> bool:
        o = self.index.pop(order_id, None)
        if o is None:
            return False
        side_book = self.bids if o.side == "buy" else self.asks
        side_book[o.price].remove(o)           # O(n) within one price level; usually tiny
        if not side_book[o.price]:
            del side_book[o.price]
        return True

    def top_of_book(self):
        return self._best_bid(), self._best_ask()

# --- tiny self-tests -------------------------------------------------
e = MatchingEngine()
assert e.submit(Order(1, "sell", 101, 10)) == []
assert e.submit(Order(2, "sell", 102,  5)) == []
# cross one level
assert e.submit(Order(3, "buy", 101, 8)) == [(3, 1, 101, 8)]
# sweep two levels + rest the remainder
trades = e.submit(Order(4, "buy", 103, 20))
assert trades == [(4, 1, 101, 2), (4, 2, 102, 5)], trades
assert e.top_of_book() == (103, None)
# cancel
assert e.cancel(4) is True
assert e.top_of_book() == (None, None)
assert e.cancel(4) is False     # already cancelled → idempotent False
print("✅ matcher: all assertions passed")

# ---- benchmark against the bad version -----------------------------
import time
e2 = MatchingEngine()
for i in range(5_000):
    e2.submit(Order(i, "sell", 100 + random.randint(0,20), 10))
start = time.perf_counter()
for i in range(5_000):
    e2.submit(Order(100_000+i, "buy", 110, 1))
print(f"BEST matcher: {time.perf_counter()-start:.3f}s for 10k events")

✅ matcher: all assertions passed
BEST matcher: 0.002s for 10k events


> 🚀 Real exchanges go further — `SortedDict` / skip lists for O(log n) price lookup, pre-allocated intrusive linked-list nodes to avoid GC pressure, etc. But the **shape** is exactly what we wrote above.

## 2. Determinism & WAL replay

Every input message is **written to an append-only file before it is matched**. After a crash we replay the file into a fresh matcher and recover the exact same state, trade for trade. This is the single most important correctness property of any exchange.

In [3]:
import io, json

# An in-memory WAL to keep the notebook self-contained (a real one is fsync'd to disk).
wal = io.StringIO()

def journal_and_submit(engine, order: Order):
    wal.write(json.dumps(order.__dict__) + "\n")
    # a real implementation would fsync() here before matching
    return engine.submit(order)

# --- live run --------------------------------------------------------
live = MatchingEngine()
inputs = [
    Order(1, "sell", 101, 10), Order(2, "sell", 102, 5),
    Order(3, "buy",  101, 8),  Order(4, "buy",  103, 20),
]
live_trades = [journal_and_submit(live, o) for o in inputs]

# --- crash! replay the WAL into a fresh engine -----------------------
replayed = MatchingEngine()
wal.seek(0)
replay_trades = [replayed.submit(Order(**json.loads(line))) for line in wal]

assert live_trades == replay_trades, "non-deterministic! 🔥"
assert live.top_of_book() == replayed.top_of_book()
print("✅ WAL replay reproduced the book exactly — identical trades")
print("   top of book:", replayed.top_of_book())

✅ WAL replay reproduced the book exactly — identical trades
   top of book: (103, None)


## 3. Market-data fan-out

One trade can be interesting to **hundreds of thousands** of subscribers. If the matcher itself writes to each subscriber socket, one slow consumer back-pressures the whole exchange — every trader pays for it.

### Bad: matcher pushes directly to subscribers

In [4]:
import time

class SlowSubscriber:
    def __init__(self): self.received = []
    def on_trade(self, t):
        time.sleep(0.002)           # 2 ms of 'network' per message
        self.received.append(t)

bad_subs = [SlowSubscriber() for _ in range(5)]

def matcher_bad(trades):
    start = time.perf_counter()
    for t in trades:
        for s in bad_subs:          # ❌ matcher is blocked on every subscriber
            s.on_trade(t)
    return time.perf_counter() - start

t = matcher_bad([f"trade-{i}" for i in range(50)])
print(f"BAD fan-out: matcher hot-path spent {t*1000:6.1f} ms delivering to subs")

BAD fan-out: matcher hot-path spent 3258.6 ms delivering to subs


### Best: matcher only puts onto a queue; a publisher thread fans out

In [5]:
import queue, threading

good_subs = [SlowSubscriber() for _ in range(5)]
feed: queue.Queue = queue.Queue()

def publisher():
    while True:
        msg = feed.get()
        if msg is None: return
        for s in good_subs:
            s.on_trade(msg)

pub = threading.Thread(target=publisher, daemon=True); pub.start()

def matcher_best(trades):
    start = time.perf_counter()
    for t in trades:
        feed.put(t)                 # ✅ O(1) enqueue, no blocking
    return time.perf_counter() - start

t = matcher_best([f"trade-{i}" for i in range(50)])
print(f"BEST fan-out: matcher hot-path spent {t*1000:6.1f} ms enqueuing")

feed.put(None); pub.join()
assert len(good_subs[0].received) == 50
print("   all subscribers received every trade — work happened off the hot path")

BEST fan-out: matcher hot-path spent    0.1 ms enqueuing


   all subscribers received every trade — work happened off the hot path


## Where to go next

- **Order types** — add `market`, `IOC`, `FOK`, and iceberg orders.
- **Self-match prevention** — cancel the resting order if the incoming trader matches themselves (required by most venues).
- **Circuit breakers** — halt trading for a symbol if price moves >X% in Y seconds; show halted status on the feed.
- **Snapshot + delta market data** — send a full book snapshot once, then only the deltas.
- **Sharding** — one matcher per symbol, spread across machines; the gateway routes by symbol.

### Further reading

- **LMAX Disruptor paper** — single-threaded design that hits millions of ops/s on one core.
- **Nasdaq ITCH 5.0 spec** — the actual wire format of a real market-data feed.
- **CME MDP 3.0** — how one of the world's largest exchanges publishes market data.
- **`04-patterns/`** in this repo — WAL, leader election, and consistent hashing are all used in production matching engines.
